In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

county = pd.read_csv(
    "../data/processed/county_table_2024.csv",
    dtype={"fips": str}
)

print("Shape:", county.shape)
print("\nMissing values:")
print(county.isna().sum())

display(county.head())

Shape: (86, 17)

Missing values:
County               0
fips                 0
visits               0
individuals          0
pounds               0
need_people_month    0
population           0
visits_per_1000      0
poverty_rate         0
snap_hh_rate         0
poor_no_snap_rate    0
med_income           0
no_vehicle_rate      0
rucc                 0
metro                0
main_food_bank       0
report_share         0
dtype: int64


,County,fips,visits,individuals,pounds,need_people_month,population,visits_per_1000,poverty_rate,snap_hh_rate,poor_no_snap_rate,med_income,no_vehicle_rate,rucc,metro,main_food_bank,report_share
0,Aitkin,27001,330.916667,719.416667,18017.975000,1411.666667,16056,20.610156,0.131130,0.089449,0.095957,60833,0.049709,8,False,Second Harvest Northland,0.969444
1,Anoka,27003,19465.083333,72999.083333,774816.833333,19215.000000,370349,52.558758,0.070194,0.071665,0.048773,101869,0.046459,1,True,Second Harvest Heartland,0.951389
2,Becker,27005,543.750000,1736.750000,51284.916667,2776.666667,35343,15.384942,0.129006,0.115753,0.064608,71388,0.057807,6,False,North Country Food Bank,1.000000
3,Beltrami,27007,1250.333333,3717.333333,86149.179167,4083.333333,46512,26.881952,0.161380,0.134658,0.082269,68975,0.058882,7,False,North Country Food Bank,0.641667
4,Benton,27009,673.083333,1693.750000,36775.416667,3062.500000,41593,16.182611,0.098657,0.085651,0.061367,74410,0.061248,3,True,Second Harvest Heartland,1.000000


In [2]:
county["food_insecure_people"] = (
    county["need_people_month"] * 12 / 7
)

county["food_insecurity_rate"] = (
    county["food_insecure_people"]
    / county["population"]
)

display(
    county[
        [
            "County",
            "population",
            "food_insecure_people",
            "food_insecurity_rate"
        ]
    ].head()
)

print(
    county["food_insecurity_rate"].describe()
)

,County,population,food_insecure_people,food_insecurity_rate
0,Aitkin,16056,2420.0,0.150722
1,Anoka,370349,32940.0,0.088943
2,Becker,35343,4760.0,0.134680
3,Beltrami,46512,7000.0,0.150499
4,Benton,41593,5250.0,0.126223


count    86.000000
mean      0.120784
std       0.019464
min       0.071430
25%       0.109671
50%       0.122509
75%       0.131462
max       0.167973
Name: food_insecurity_rate, dtype: float64


In [3]:
baseline_intercept = 0.08
baseline_slope = 0.34

county["benchmark_use_rate"] = (
    baseline_intercept
    +
    baseline_slope
    * county["food_insecurity_rate"]
)

county["expected_unscaled"] = (
    county["population"]
    * county["benchmark_use_rate"]
)

In [4]:
#scaling expected activity to the statewide observed total
statewide_observed = county["visits"].sum()

statewide_expected_unscaled = (
    county["expected_unscaled"].sum()
)

scale_factor = (
    statewide_observed
    / statewide_expected_unscaled
)

county["expected_visits"] = (
    county["expected_unscaled"]
    * scale_factor
)

county["coverage_ratio"] = (
    county["visits"]
    / county["expected_visits"]
)

print(
    "Observed statewide visits:",
    statewide_observed
)

print(
    "Expected statewide visits:",
    county["expected_visits"].sum()
)

print(
    "Scale factor:",
    scale_factor
)

Observed statewide visits: 236887.91666666666
Expected statewide visits: 236887.9166666667
Scale factor: 0.351039754051977


In [5]:
checkpoint = county[
    county["County"].isin(
        ["Scott", "Steele"]
    )
][
    [
        "County",
        "food_insecurity_rate",
        "visits",
        "expected_visits",
        "coverage_ratio"
    ]
]

display(checkpoint)

,County,food_insecurity_rate,visits,expected_visits,coverage_ratio
68,Scott,0.071430,630.416667,5658.114922,0.111418
73,Steele,0.109511,4932.666667,1540.755605,3.201459


In [6]:
lowest_8 = (
    county[
        [
            "County",
            "visits",
            "expected_visits",
            "coverage_ratio"
        ]
    ]
    .sort_values("coverage_ratio")
    .head(8)
)

highest_5 = (
    county[
        [
            "County",
            "visits",
            "expected_visits",
            "coverage_ratio"
        ]
    ]
    .sort_values(
        "coverage_ratio",
        ascending=False
    )
    .head(5)
)

print("Lowest 8:")
display(lowest_8)

print("Highest 5:")
display(highest_5)

Lowest 8:


,County,visits,expected_visits,coverage_ratio
68,Scott,630.416667,5658.114922,0.111418
60,Pope,71.250000,473.426254,0.150499
51,Nicollet,236.083333,1424.687821,0.165709
53,Norman,45.500000,270.707817,0.168078
36,Lac qui Parle,50.250000,277.166948,0.181299
74,Stevens,110.916667,425.081059,0.260931
7,Brown,293.083333,1062.316504,0.275891
67,Roseau,183.583333,644.565155,0.284817


Highest 5:


,County,visits,expected_visits,coverage_ratio
73,Steele,4932.666667,1540.755605,3.201459
81,Washington,24896.000000,10440.708615,2.384512
65,Rice,4656.166667,2801.283196,1.662155
54,Olmsted,10776.166667,6587.766482,1.635785
26,Hennepin,80605.750000,52738.134493,1.528415


In [7]:

#sensitivity analysis - changing from 0.08+0.34FI to 0.05+0.50FI.
alternative_intercept = 0.05
alternative_slope = 0.50

county["alt_use_rate"] = (
    alternative_intercept
    +
    alternative_slope
    * county["food_insecurity_rate"]
)

county["alt_expected_unscaled"] = (
    county["population"]
    * county["alt_use_rate"]
)

alternative_scale = (
    county["visits"].sum()
    /
    county["alt_expected_unscaled"].sum()
)

county["alt_expected_visits"] = (
    county["alt_expected_unscaled"]
    * alternative_scale
)

county["alt_coverage_ratio"] = (
    county["visits"]
    /
    county["alt_expected_visits"]
)

sensitivity_rank_correlation = (
    county[
        [
            "coverage_ratio",
            "alt_coverage_ratio"
        ]
    ]
    .corr(method="spearman")
    .iloc[0, 1]
)

print(
    "Sensitivity rank correlation:",
    sensitivity_rank_correlation
)

#quantifying actual rank movement
county["baseline_rank"] = (
    county["coverage_ratio"].rank()
)

county["alternative_rank"] = (
    county["alt_coverage_ratio"].rank()
)

county["rank_change"] = (
    county["baseline_rank"]
    -
    county["alternative_rank"]
).abs()

display(
    county[
        [
            "County",
            "coverage_ratio",
            "alt_coverage_ratio",
            "baseline_rank",
            "alternative_rank",
            "rank_change"
        ]
    ]
    .sort_values(
        "rank_change",
        ascending=False
    )
    .head(10)
)


Sensitivity rank correlation: 0.9970187272984574


,County,coverage_ratio,alt_coverage_ratio,baseline_rank,alternative_rank,rank_change
84,Wright,0.407869,0.436157,23.0,31.0,8.0
12,Chisago,0.528789,0.551296,42.0,47.0,5.0
3,Beltrami,0.583810,0.547743,49.0,45.0,4.0
18,Dakota,0.576029,0.595478,48.0,52.0,4.0
66,Rock,0.395408,0.399872,19.0,23.0,4.0
14,Clearwater,0.489351,0.458875,38.0,34.0,4.0
9,Carver,0.503870,0.540424,40.0,44.0,4.0
0,Aitkin,0.447342,0.419575,32.0,29.0,3.0
44,Martin,0.539119,0.503581,43.0,40.0,3.0
16,Cottonwood,0.626545,0.594851,54.0,51.0,3.0


In [8]:
per_capita_rank_correlation = (
    county[
        [
            "coverage_ratio",
            "visits_per_1000"
        ]
    ]
    .corr(method="spearman")
    .iloc[0, 1]
)

print(
    "Coverage ratio vs visits per 1,000 "
    "rank correlation:",
    per_capita_rank_correlation
)

Coverage ratio vs visits per 1,000 rank correlation: 0.9921694419548092


Interpretation. The coverage-ratio ranking is almost identical to the ranking based on food-shelf visits per capita (Spearman correlation approximately 0.99). This indicates that county-to-county differences in estimated food insecurity are relatively small compared with the much larger geographic differences in recorded food-shelf use. Therefore, variation in activity is likely being driven substantially by factors beyond measured need, including accessibility, food-shelf supply, reporting, food-bank structure, and cross-county utilization.
Scott County has one of the lowest coverage ratios, but this does not by itself establish that Scott is underserved. Because Scott borders large neighboring counties, residents may obtain food from shelves outside their county. A spatial accessibility analysis should therefore allow residents to reach nearby shelves across county boundaries. A 2SFCA or distance/drive-time analysis can test whether Scott has genuinely poor access or whether its low recorded activity reflects cross-county service us

In [9]:
county.to_csv(
    "../data/processed/coverage_ratio_2024.csv",
    index=False
)

print("Saved:", county.shape)

Saved: (86, 30)
